In [1]:
import pandas as pd
import numpy as np

df = pd.read_excel("/kaggle/input/datasets/karvysingh/dalisa/FCC_feed_data.xlsx",
                   sheet_name="Feed Data")

# remove the units row
df = df[pd.to_numeric(df["Sample No."], errors="coerce").notna()].copy()

df["Sample No."] = df["Sample No."].astype(int)

for c in df.columns[1:]:
    df[c] = pd.to_numeric(df[c])

df.head()

,Sample No.,Sp. Gr,CCR,T.S,T.N,IBP,5 %,10 %,30 %,50 %,70 %,90 %,95 %,FBP,P,N,A
1,1,0.9245,0.40,2.93,1401,37,317,342,392,423,454,498,514,538,8.3,24.1,67.6
2,2,0.9960,1.55,4.35,200,42,409,428,458,477,494,514,523,537,7.8,25.3,66.8
3,3,0.8683,0.04,0.03,22,103,224,263,337,380,420,473,497,536,15.2,45.0,39.8
4,4,0.9136,0.04,1.54,329,111,276,318,368,391,407,427,436,460,14.8,15.5,70.5
5,5,0.9303,0.28,3.06,1161,169,257,320,402,438,474,519,535,574,11.2,23.3,65.6


In [2]:
features = [
    "Sp. Gr",
    "5 %",
    "10 %",
    "30 %",
    "50 %",
    "70 %",
    "90 %",
    "95 %"
]

targets = ["P", "N", "A"]

# published validation samples
val = df[df["Sample No."].between(17, 22)].copy()

# remaining samples for development
dev = df[~df["Sample No."].between(17, 22)].copy()

X = dev[features]
y = dev[targets]

X_val = val[features]
y_val = val[targets]

In [3]:
train = df[df["Sample No."].between(1, 16)].copy()
test  = df[df["Sample No."].between(23, 28)].copy()
val   = df[df["Sample No."].between(17, 22)].copy()

X_train = train[features]
y_train = train[targets]

X_test = test[features]
y_test = test[targets]

X_val = val[features]
y_val = val[targets]

In [4]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler(feature_range=(-1, 1))

X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)
X_val   = scaler.transform(X_val)

In [5]:
import torch
import torch.nn as nn
import numpy as np

torch.manual_seed(42)

Xtr = torch.tensor(X_train, dtype=torch.float32)
Xte = torch.tensor(X_test, dtype=torch.float32)
Xv  = torch.tensor(X_val, dtype=torch.float32)

class ANN(nn.Module):
    def __init__(self, h1, h2):
        super().__init__()
        self.fc1 = nn.Linear(8, h1)
        self.fc2 = nn.Linear(h1, h2)
        self.out = nn.Linear(h2, 1)

    def forward(self, x):
        x = torch.tanh(self.fc1(x))      # tansig
        x = torch.sigmoid(self.fc2(x))  # logsig
        return self.out(x)               # purelin

In [6]:
def train_model(y, h1, h2, seed):
    torch.manual_seed(seed)

    model = ANN(h1, h2)

    y = torch.tensor(
        np.asarray(y, dtype=np.float32).reshape(-1, 1)
    )

    optimizer = torch.optim.LBFGS(
        model.parameters(),
        lr=0.5,
        max_iter=1000,
        tolerance_grad=1e-9,
        tolerance_change=1e-12,
        line_search_fn="strong_wolfe"
    )

    loss_fn = nn.MSELoss()

    def closure():
        optimizer.zero_grad()
        pred = model(Xtr)
        loss = loss_fn(pred, y)
        loss.backward()
        return loss

    optimizer.step(closure)

    return model

In [7]:
from sklearn.metrics import mean_squared_error

records = []

for seed in range(50):

    model_P = train_model(y_train["P"], 12, 13, seed)
    model_N = train_model(y_train["N"], 12, 12, seed + 100)
    model_A = train_model(y_train["A"], 9, 9, seed + 200)

    with torch.no_grad():
        p = model_P(Xte).numpy().ravel()
        n = model_N(Xte).numpy().ravel()
        a = model_A(Xte).numpy().ravel()

    total = p + n + a

    pred_test = np.column_stack([
        100*p/total,
        100*n/total,
        100*a/total
    ])

    obs_test = y_test[["P", "N", "A"]].values

    rmse_test = np.sqrt(
        mean_squared_error(
            obs_test.ravel(),
            pred_test.ravel()
        )
    )

    records.append((seed, rmse_test))

records.sort(key=lambda x: x[1])

records[:10]

[(34, np.float64(6.692781798922146)),
 (37, np.float64(6.987325060523674)),
 (20, np.float64(8.01112232165701)),
 (21, np.float64(9.31565707089622)),
 (29, np.float64(9.587134080683542)),
 (40, np.float64(9.645340052166741)),
 (19, np.float64(9.787056683376843)),
 (42, np.float64(9.822052984891236)),
 (35, np.float64(10.077536837870042)),
 (43, np.float64(10.348222825805209))]

In [8]:
records.sort(key=lambda x: x[1])

best_seed = records[0][0]

print("Best seed:", best_seed)
print("Best test RMSE:", records[0][1])

model_P = train_model(
    y_train["P"], 12, 13, best_seed
)

model_N = train_model(
    y_train["N"], 12, 12, best_seed + 100
)

model_A = train_model(
    y_train["A"], 9, 9, best_seed + 200
)

with torch.no_grad():

    P_raw = model_P(Xv).numpy().ravel()
    N_raw = model_N(Xv).numpy().ravel()
    A_raw = model_A(Xv).numpy().ravel()

total = P_raw + N_raw + A_raw

P = 100 * P_raw / total
N = 100 * N_raw / total
A = 100 * A_raw / total

pred = np.column_stack([P, N, A])

Best seed: 34
Best test RMSE: 6.692781798922146


In [9]:
result = pd.DataFrame({
    "Sample": val["Sample No."].values,
    "P_exp": y_val["P"].values,
    "P_pred": P,
    "N_exp": y_val["N"].values,
    "N_pred": N,
    "A_exp": y_val["A"].values,
    "A_pred": A,
})

display(result.round(2))

,Sample,P_exp,P_pred,N_exp,N_pred,A_exp,A_pred
0,17,15.8,12.39,35.9,33.919998,48.3,53.689999
1,18,14.4,8.42,23.3,22.110001,62.2,69.470001
2,19,4.9,8.42,21.1,24.610001,74.0,66.959999
3,20,10.9,10.90,29.9,27.040001,59.2,62.060001
4,21,10.4,10.97,18.5,22.330000,71.0,66.690002
5,22,6.3,8.46,26.2,24.440001,67.5,67.099998


In [10]:
def predict_pna(X):
    with torch.no_grad():
        p = model_P(X).numpy().ravel()
        n = model_N(X).numpy().ravel()
        a = model_A(X).numpy().ravel()

    s = p + n + a

    p = 100 * p / s
    n = 100 * n / s
    a = 100 * a / s

    return np.column_stack([p, n, a])


from sklearn.metrics import mean_squared_error, r2_score

def evaluate(name, X, y):
    pred = predict_pna(X)
    obs = y[["P", "N", "A"]].values

    rmse = np.sqrt(
        mean_squared_error(obs.ravel(), pred.ravel())
    )

    r2 = r2_score(
        obs.ravel(),
        pred.ravel()
    )

    print(f"{name:12s} RMSE = {rmse:.3f}   R² = {r2:.4f}")


evaluate("TRAIN", Xtr, y_train)
evaluate("TEST", Xte, y_test)
evaluate("VALIDATION", Xv, y_val)

TRAIN        RMSE = 0.574   R² = 0.9993
TEST         RMSE = 6.693   R² = 0.8848
VALIDATION   RMSE = 3.850   R² = 0.9726
